# Cell 1 — Notebook title

# GNN4EEG FACED HetEmotionNet — cross-9 processed FACED dataset quick check

### Dataset-first, numbered, Kaggle-ready, checkpointed, and resumable

This notebook trains the **GNN4EEG EEG-only HetEmotionNet implementation** as a **nine-class multiclass classifier** using subject-wise processed FACED feature files such as:

```text
sub001.pkl
sub009.pkl.pkl
```

Each accepted subject file is canonicalized to `(28 videos, 32 channels, 30 one-second windows, 5 bands)`. By default, the final two channels are removed to match the GNN4EEG benchmark, giving 30 EEG graph nodes.

**Fixed experiment in this notebook:** `cross-9`.


> **Quick-check build:** 2 outer folds, 2 inner folds, one hyperparameter combination, and 2 epochs. Use this only to verify that loading, preprocessing, graph construction, training, checkpointing, evaluation, and resume logic work.

# Cell 2 — Scope, input mapping, and fidelity note

## Processed PKL input used here

Accepted feature shape:

```text
(28 videos, 32 channels, 30 seconds, 5 DE/PSD bands)
```

The notebook prepares one training sample for every one-second window:

- **Frequency stream `x_F`:** current second across five bands, `(30 nodes, 5 bands)`.
- **Temporal stream `x_T`:** complete 30-second clip averaged across bands, `(30 nodes, 30 seconds)`.
- **Adjacency `A`:** one clip-level 30×30 graph, reused by the 30 windows of that clip.
- **Output:** nine logits for Anger, Disgust, Fear, Sadness, Neutral, Amusement, Inspiration, Joy, and Tenderness.

The model follows the public GNN4EEG `Het_Model` design: graph propagation, bidirectional GRU processing for frequency and temporal streams, feature fusion, and a nine-class linear classifier. It is the benchmark's EEG-only adaptation, not the original multimodal EEG/ECG/GSR network.

This quick-check build uses **2 outer folds, 2 inner folds, hidden dimension `20`, learning rate `1e-3`, and 2 epochs**. It is designed only to confirm that the complete pipeline runs correctly. Its accuracy must **not** be used as the final thesis or paper result.


# Cell 3 — Kaggle input checklist

## Required input

Attach the authors' processed FACED feature dataset (6 GB). The dataset contains subject-wise .pkl feature containers because the authors converted the original raw 2 GB EEG data into compact processed feature files. Valid files must contain a numeric array that can be mapped to:

```text
(28, 32, 30, 5)
```

The loader also accepts already-trimmed `(28, 30, 30, 5)` arrays. It searches recursively under `/kaggle/input`, supports `.pkl` and `.pkl.pkl`, ignores raw `(28, 32, 7500)` EEG files, and rejects unrelated pickle files.

## Resume after the Kaggle runtime limit

At the end of each session, download the generated `cross_9` quick-check resume ZIP from `/kaggle/working`. Upload it as a private Kaggle dataset and attach it to the next session. The notebook restores cached features, splits, adjacency progress, epoch checkpoints, histories, completed-fold markers, predictions, and results automatically.

## GPU

Select **Kaggle → Settings → Accelerator → GPU** before training.


# Cell 4 — Execution roadmap

## Notebook sections

1. Imports and environment
2. Discover, validate, and cache processed subject PKL files
3. Configuration and checkpoint workspace
4. Subject normalization and two-stream preparation
5. Clip-level adjacency construction
6. Exact outer/inner split construction
7. PyTorch datasets and data loaders
8. GNN4EEG HetEmotionNet model
9. Training, metrics, and epoch-level resume utilities
10. Nested cross-validation runner
11. Results, inference, and resume export

Run the notebook from top to bottom. Use `RUN_PROFILE = "smoke_test"` for a short pipeline check, then switch to `"paper_ncv"` for the full experiment. Smoke-test and paper-mode outputs are kept in separate folders.


# Cell 5 — Section 1 start — Imports and environment

# Section 1 — Imports and environment

The next cell is the single import cell. No later section introduces hidden imports.


In [ ]:
# Cell 6 — Import all libraries

import os
import gc
import re
import json
import math
import time
import copy
import pickle
import random
import shutil
import zipfile
import hashlib
import warnings
import importlib.util
from itertools import product, permutations
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Optional, Sequence, Tuple

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm

from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
    confusion_matrix,
    mutual_info_score,
)

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

warnings.filterwarnings('ignore', category=UserWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 170)
print('All imports completed successfully.')


In [ ]:
# Cell 7 — Verify packages, seed everything, and select device

REQUIRED_MODULES = ['numpy', 'pandas', 'sklearn', 'torch', 'matplotlib', 'tqdm']
missing = [name for name in REQUIRED_MODULES if importlib.util.find_spec(name) is None]
if missing:
    raise ImportError('Missing required modules: ' + ', '.join(missing))

def seed_everything(seed: int = 42) -> None:
    os.environ['PYTHONHASHSEED'] = str(seed)
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

SEED = 42
seed_everything(SEED)
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

environment = pd.DataFrame({
    'component': ['Python', 'NumPy', 'Pandas', 'PyTorch', 'CUDA available', 'Device'],
    'value': [
        os.sys.version.split()[0], np.__version__, pd.__version__, torch.__version__,
        str(torch.cuda.is_available()),
        torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU only',
    ],
})
display(environment)


# Cell 8 — Section 1 end — Imports complete

**Section 1 complete.**


# Cell 9 — Section 2 start — Load processed FACED PKL features first

# Section 2 — Discover, validate, and cache processed subject files

The next cells restore an attached resume bundle before scanning the dataset. A valid cached tensor is reused immediately, so expensive PKL loading does not repeat after a Kaggle restart.


### Dataset handling note
This notebook does not use the original 2 GB raw FACED EEG recordings because they are too large for Kaggle execution. It uses the authors' processed FACED dataset (~6 GB), where each subject is stored as a `.pkl` feature file. These `.pkl` files are only containers for the already processed dataset features; the experiment remains a FACED dataset implementation.


In [ ]:
# Cell 10 — Establish the resumable workspace and discover candidate PKL files

TASK_SLUG = 'hetemotionnet_processed_FACED_cross_9_full_resumable'
KAGGLE_WORKING = Path('/kaggle/working')
WORK_ROOT_PRESET = (KAGGLE_WORKING / TASK_SLUG) if KAGGLE_WORKING.exists() else (Path.cwd() / TASK_SLUG)
BOOTSTRAP_CACHE_DIR = WORK_ROOT_PRESET / 'cache'
BOOTSTRAP_CACHE_DIR.mkdir(parents=True, exist_ok=True)

# Kaggle input location of the authors' processed FACED dataset.
# Example:
# PKL_ROOT_OVERRIDE = '/kaggle/input/faced-processed-dataset'
PKL_ROOT_OVERRIDE = None
DROP_LAST_TWO_CHANNELS = True
EXPECTED_VIDEOS = 28
EXPECTED_SECONDS = 30
EXPECTED_BANDS = 5
EXPECTED_INPUT_CHANNELS = (32, 30)

def restore_bootstrap_resume_bundle() -> Optional[Path]:
    search_root = Path('/kaggle/input')
    if not search_root.exists():
        return None
    patterns = [
        '*cross_9*quickcheck*resume*.zip',
        '*cross_9*quickcheck*resume*.zip',
        '*hetemotionnet*processed*pkl*quickcheck*resume*.zip',
    ]
    candidates = []
    for pattern in patterns:
        candidates.extend(search_root.rglob(pattern))
    candidates = sorted(set(candidates), key=lambda path: path.stat().st_mtime, reverse=True)
    if not candidates:
        print('No attached resume bundle found.')
        return None
    bundle = candidates[0]
    print('Restoring resume bundle:', bundle)
    with zipfile.ZipFile(bundle, 'r') as archive:
        archive.extractall(WORK_ROOT_PRESET.parent)
    return bundle

RESTORED_BUNDLE = restore_bootstrap_resume_bundle()

def candidate_pickle_files(explicit_root: Optional[str] = None) -> List[Path]:
    roots = [Path(explicit_root)] if explicit_root else [Path('/kaggle/input'), Path.cwd()]
    files = []
    for root in roots:
        if root.exists():
            files.extend(path for path in root.rglob('*.pkl*') if path.is_file())
    # Do not scan restored working checkpoints as dataset inputs.
    files = [path for path in files if WORK_ROOT_PRESET not in path.parents]
    return sorted(set(files))

CANDIDATE_PKL_FILES = candidate_pickle_files(PKL_ROOT_OVERRIDE)
print('Processed FACED feature files discovered:', len(CANDIDATE_PKL_FILES))
for path in CANDIDATE_PKL_FILES[:10]:
    print(' -', path)


In [ ]:
# Cell 11 — Define robust PKL extraction and shape canonicalization

SUBJECT_PATTERN = re.compile(r'(?i)(?:sub(?:ject)?)[_-]?(\d+)')

def subject_number_from_path(path: Path) -> int:
    match = SUBJECT_PATTERN.search(path.name)
    if match:
        return int(match.group(1))
    numbers = re.findall(r'\d+', path.stem)
    if numbers:
        return int(numbers[-1])
    raise ValueError(f'No subject number could be parsed from {path.name!r}.')

def iter_numeric_arrays(value: Any, depth: int = 0) -> Iterable[np.ndarray]:
    if depth > 4:
        return
    if isinstance(value, np.ndarray):
        if np.issubdtype(value.dtype, np.number):
            yield value
        elif value.dtype == object:
            for item in value.flat:
                yield from iter_numeric_arrays(item, depth + 1)
    elif torch.is_tensor(value):
        yield value.detach().cpu().numpy()
    elif isinstance(value, dict):
        for item in value.values():
            yield from iter_numeric_arrays(item, depth + 1)
    elif isinstance(value, (list, tuple)):
        for item in value:
            yield from iter_numeric_arrays(item, depth + 1)

def canonicalize_subject_feature(array: np.ndarray) -> Tuple[np.ndarray, Tuple[int, ...]]:
    array = np.squeeze(np.asarray(array))
    if array.ndim != 4:
        raise ValueError(f'not a four-dimensional feature array: {array.shape}')

    targets = [(28, 32, 30, 5), (28, 30, 30, 5)]
    for target in targets:
        if tuple(array.shape) == target:
            canonical = array
            break
    else:
        canonical = None
        for target in targets:
            for order in permutations(range(4)):
                if tuple(array.shape[index] for index in order) == target:
                    canonical = array.transpose(order)
                    break
            if canonical is not None:
                break
        if canonical is None:
            raise ValueError(f'cannot map shape {array.shape} to (28,32,30,5) or (28,30,30,5)')

    original_shape = tuple(int(value) for value in canonical.shape)
    canonical = np.asarray(canonical, dtype=np.float32)
    if canonical.shape[1] == 32 and DROP_LAST_TWO_CHANNELS:
        canonical = canonical[:, :30, :, :]
    if canonical.shape != (EXPECTED_VIDEOS, 30, EXPECTED_SECONDS, EXPECTED_BANDS):
        raise ValueError(f'canonical output has unexpected shape {canonical.shape}')
    if not np.isfinite(canonical).all():
        raise ValueError('contains NaN or infinity')
    return np.ascontiguousarray(canonical), original_shape

def read_feature_from_pickle(path: Path) -> Tuple[np.ndarray, Tuple[int, ...]]:
    with open(path, 'rb') as handle:
        value = pickle.load(handle)
    errors = []
    candidates = sorted(iter_numeric_arrays(value), key=lambda array: array.size, reverse=True)
    for array in candidates:
        try:
            return canonicalize_subject_feature(array)
        except Exception as exc:
            errors.append(str(exc))
    preview = ' | '.join(errors[:3]) if errors else 'no numeric arrays found'
    raise ValueError(preview)

print('PKL extraction functions are ready.')


In [ ]:
# Cell 12 — Load accepted subject features or reuse the atomic dataset cache

FEATURE_CACHE_PATH = BOOTSTRAP_CACHE_DIR / 'processed_subject_features_30ch.npy'
DATASET_META_PATH = BOOTSTRAP_CACHE_DIR / 'processed_subject_features_metadata.json'
FILE_MANIFEST_PATH = BOOTSTRAP_CACHE_DIR / 'processed_subject_file_manifest.csv'

def atomic_save_npy(path: Path, array: np.ndarray) -> None:
    temporary = path.with_suffix(path.suffix + '.tmp')
    with open(temporary, 'wb') as handle:
        np.save(handle, array)
    temporary.replace(path)

if FEATURE_CACHE_PATH.exists() and DATASET_META_PATH.exists() and FILE_MANIFEST_PATH.exists():
    feature_5d = np.load(FEATURE_CACHE_PATH, mmap_mode='r')
    with open(DATASET_META_PATH) as handle:
        dataset_meta = json.load(handle)
    subject_file_numbers = np.asarray(dataset_meta['subject_file_numbers'], dtype=np.int64)
    file_manifest = pd.read_csv(FILE_MANIFEST_PATH)
    print('Loaded processed feature cache:', FEATURE_CACHE_PATH)
else:
    accepted_by_subject: Dict[int, Dict[str, Any]] = {}
    rejected_rows = []
    for path in tqdm(CANDIDATE_PKL_FILES, desc='Inspect processed PKL files'):
        try:
            subject_number = subject_number_from_path(path)
            feature, original_shape = read_feature_from_pickle(path)
            record = {
                'subject_number': subject_number,
                'path': str(path),
                'file_name': path.name,
                'file_size_bytes': int(path.stat().st_size),
                'original_feature_shape': str(original_shape),
                'canonical_shape': str(tuple(feature.shape)),
                'feature': feature,
            }
            if subject_number in accepted_by_subject:
                previous = accepted_by_subject[subject_number]
                previous_key = (previous['file_size_bytes'], previous['path'])
                current_key = (record['file_size_bytes'], record['path'])
                if current_key > previous_key:
                    rejected_rows.append({
                        'path': previous['path'],
                        'status': 'duplicate_skipped',
                        'reason': f'replaced by {path.name}',
                    })
                    accepted_by_subject[subject_number] = record
                else:
                    rejected_rows.append({
                        'path': str(path),
                        'status': 'duplicate_skipped',
                        'reason': (
                            f'subject {subject_number} already loaded from '
                            f'{Path(previous["path"]).name}'
                        ),
                    })
            else:
                accepted_by_subject[subject_number] = record
        except Exception as exc:
            rejected_rows.append({
                'path': str(path),
                'status': 'rejected',
                'reason': str(exc)[:300],
            })

    if not accepted_by_subject:
        examples = '\n'.join(
            f' - {row["path"]}: {row["reason"]}' for row in rejected_rows[:10]
        )
        raise FileNotFoundError(
            'No valid processed FACED feature PKL file was found. Expected an array mappable to '
            '(28,32,30,5). Rejection examples:\n' + examples
        )

    ordered_records = [accepted_by_subject[key] for key in sorted(accepted_by_subject)]
    feature_5d_array = np.stack(
        [record.pop('feature') for record in ordered_records], axis=0
    ).astype(np.float32)
    subject_file_numbers = np.asarray(
        [record['subject_number'] for record in ordered_records], dtype=np.int64
    )

    selected_rows = [
        {**record, 'status': 'accepted', 'reason': ''} for record in ordered_records
    ]
    file_manifest = pd.DataFrame(selected_rows + rejected_rows)
    file_manifest.to_csv(FILE_MANIFEST_PATH, index=False)

    fingerprint_text = '|'.join(
        f"{row['subject_number']}:{Path(row['path']).name}:{row['file_size_bytes']}"
        for row in ordered_records
    )
    dataset_fingerprint = hashlib.sha256(fingerprint_text.encode()).hexdigest()
    dataset_meta = {
        'subject_file_numbers': subject_file_numbers.tolist(),
        'num_subjects': int(len(subject_file_numbers)),
        'feature_shape': list(feature_5d_array.shape),
        'drop_last_two_channels': bool(DROP_LAST_TWO_CHANNELS),
        'dataset_fingerprint': dataset_fingerprint,
    }
    atomic_save_npy(FEATURE_CACHE_PATH, feature_5d_array)
    with open(DATASET_META_PATH, 'w') as handle:
        json.dump(dataset_meta, handle, indent=2)
    del feature_5d_array
    gc.collect()
    feature_5d = np.load(FEATURE_CACHE_PATH, mmap_mode='r')
    print('Saved atomic processed feature cache:', FEATURE_CACHE_PATH)

DATASET_FINGERPRINT = dataset_meta['dataset_fingerprint']
print('Accepted subject count:', feature_5d.shape[0])
print('Canonical feature shape:', feature_5d.shape)
print('Dataset fingerprint:', DATASET_FINGERPRINT[:16])
display(file_manifest[file_manifest['status'] == 'accepted'].head(10))
if (file_manifest['status'] != 'accepted').any():
    display(file_manifest[file_manifest['status'] != 'accepted'].head(10))

# Convert (subject, video, node, second, band) to the benchmark's
# (subject, 840 one-second samples, 30 nodes × 5 bands) layout.
NUM_SUBJECTS = int(feature_5d.shape[0])
data_3d = np.asarray(feature_5d).transpose(0, 1, 3, 2, 4).reshape(
    NUM_SUBJECTS, 840, 150
)
data_3d = np.ascontiguousarray(data_3d, dtype=np.float32)
EXPECTED_SHAPE = (NUM_SUBJECTS, 840, 150)
assert data_3d.shape == EXPECTED_SHAPE
print('Benchmark-compatible flattened shape:', data_3d.shape)


In [ ]:
# Cell 13 — Create exact nine-class labels and sample metadata

EMOTION_NAMES = [
    'Anger', 'Disgust', 'Fear', 'Sadness', 'Neutral',
    'Amusement', 'Inspiration', 'Joy', 'Tenderness',
]
VIDEO_LABELS = np.array(
    [0] * 3 + [1] * 3 + [2] * 3 + [3] * 3 + [4] * 4 +
    [5] * 3 + [6] * 3 + [7] * 3 + [8] * 3,
    dtype=np.int64,
)
assert len(VIDEO_LABELS) == 28

NUM_VIDEOS = 28
SECONDS_PER_VIDEO = 30
NUM_NODES = 30
NUM_BANDS = 5
NUM_CLASSES = 9
SAMPLES_PER_SUBJECT = NUM_VIDEOS * SECONDS_PER_VIDEO

sample_labels = np.tile(np.repeat(VIDEO_LABELS, SECONDS_PER_VIDEO), NUM_SUBJECTS)
subject_ids = np.repeat(np.arange(NUM_SUBJECTS, dtype=np.int64), SAMPLES_PER_SUBJECT)
original_subject_numbers = subject_file_numbers[subject_ids]
video_ids = np.tile(np.repeat(np.arange(NUM_VIDEOS), SECONDS_PER_VIDEO), NUM_SUBJECTS)
second_ids = np.tile(np.arange(SECONDS_PER_VIDEO), NUM_SUBJECTS * NUM_VIDEOS)
clip_ids = np.repeat(np.arange(NUM_SUBJECTS * NUM_VIDEOS), SECONDS_PER_VIDEO)

metadata_preview = pd.DataFrame({
    'sample_index': np.arange(12),
    'subject_position': subject_ids[:12],
    'subject_file_number': original_subject_numbers[:12],
    'video': video_ids[:12],
    'second': second_ids[:12],
    'clip_id': clip_ids[:12],
    'label': sample_labels[:12],
    'emotion': [EMOTION_NAMES[index] for index in sample_labels[:12]],
})
display(metadata_preview)


In [ ]:
# Cell 14 — Run dataset quality, duplication, and class-balance checks

quality = pd.DataFrame({
    'check': [
        'feature_5d shape', 'flattened shape', 'finite values', 'NaN count', 'Inf count',
        'accepted subjects', 'unique subject file numbers', 'one-second samples', 'classes',
    ],
    'value': [
        str(tuple(feature_5d.shape)), str(data_3d.shape), bool(np.isfinite(data_3d).all()),
        int(np.isnan(data_3d).sum()), int(np.isinf(data_3d).sum()), NUM_SUBJECTS,
        int(np.unique(subject_file_numbers).size), int(sample_labels.size), int(np.unique(sample_labels).size),
    ],
})
display(quality)

if np.unique(subject_file_numbers).size != NUM_SUBJECTS:
    raise ValueError('Duplicate subject numbers remain after file selection.')
if not np.isfinite(data_3d).all():
    raise ValueError('Processed features contain NaN or infinity values.')

class_counts = pd.DataFrame({
    'class_id': np.arange(NUM_CLASSES),
    'emotion': EMOTION_NAMES,
    'videos_per_subject': np.bincount(VIDEO_LABELS, minlength=NUM_CLASSES),
    'one_second_samples': np.bincount(sample_labels, minlength=NUM_CLASSES),
})
display(class_counts)


# Cell 15 — Section 2 end — Processed dataset understood and cached

The subject files have been filtered by actual tensor shape, canonicalized, deduplicated, trimmed to 30 EEG nodes, and stored in an atomic cache. Raw `(28,32,7500)` files are not used by this notebook.


# Cell 16 — Section 3 start — Configuration and checkpoint workspace

# Section 3 — Experiment configuration and automatic resume for full nested-CV training

`paper_ncv` uses the full 10 × 3 nested-CV procedure and the complete hidden-dimension/learning-rate grid. It is computationally expensive and is intended to continue across Kaggle sessions through checkpoints.


In [ ]:
# Cell 17 — Define the fixed-task configuration

RUN_PROFILE = 'paper_ncv'  # use 'smoke_test' first, then return to 'paper_ncv'

@dataclass
class Config:
    protocol: str = 'cross_subject'
    run_profile: str = RUN_PROFILE
    seed: int = 42

    num_subjects: int = NUM_SUBJECTS
    num_videos: int = 28
    seconds_per_video: int = 30
    num_nodes: int = 30
    num_bands: int = 5
    num_classes: int = 9

    outer_folds: int = 10
    inner_folds: int = 3
    hidden_grid: Tuple[int, ...] = (20, 40, 80)
    learning_rate_grid: Tuple[float, ...] = (1e-4, 1e-3, 1e-2)
    max_epochs: int = 100
    dropout: float = 0.5
    batch_size: int = 256
    l1_reg: float = 0.005
    l2_reg: float = 0.005
    gradient_clip: float = 20.0
    num_workers: int = 2
    mixed_precision: bool = True
    adjacency_method: str = 'mutual_information'  # 'correlation' is faster for diagnosis
    adjacency_threshold: float = 0.7
    mi_bins: int = 10

    normalization: str = 'paper_subject_zscore'
    folds_to_run: Optional[Tuple[int, ...]] = None
    work_root: str = str(WORK_ROOT_PRESET)

    @property
    def effective_hidden_grid(self) -> Tuple[int, ...]:
        return (20,) if self.run_profile == 'smoke_test' else self.hidden_grid

    @property
    def effective_lr_grid(self) -> Tuple[float, ...]:
        return (1e-3,) if self.run_profile == 'smoke_test' else self.learning_rate_grid

    @property
    def effective_max_epochs(self) -> int:
        return 3 if self.run_profile == 'smoke_test' else self.max_epochs

    @property
    def effective_folds(self) -> Tuple[int, ...]:
        if self.folds_to_run is not None:
            return self.folds_to_run
        return (0,) if self.run_profile == 'smoke_test' else tuple(range(self.outer_folds))

CFG = Config()
assert CFG.protocol == 'cross_subject'
assert CFG.run_profile in {'paper_ncv', 'smoke_test'}
assert CFG.adjacency_method in {'mutual_information', 'correlation'}
if CFG.protocol == 'cross_subject' and NUM_SUBJECTS < CFG.outer_folds:
    raise ValueError('Cross-9 needs at least 10 accepted subjects for 10-fold evaluation.')

display(pd.DataFrame([{'parameter': key, 'value': value} for key, value in asdict(CFG).items()]))
print('Effective hidden grid:', CFG.effective_hidden_grid)
print('Effective learning-rate grid:', CFG.effective_lr_grid)
print('Effective maximum epochs:', CFG.effective_max_epochs)
print('Outer folds scheduled:', CFG.effective_folds)


In [ ]:
# Cell 18 — Create profile-separated cache, split, checkpoint, and result folders

WORK_ROOT = Path(CFG.work_root)
PROFILE_ROOT = WORK_ROOT / CFG.run_profile
CACHE_DIR = WORK_ROOT / 'cache'
SPLIT_DIR = WORK_ROOT / 'splits'
CHECKPOINT_DIR = PROFILE_ROOT / 'checkpoints'
RESULT_DIR = PROFILE_ROOT / 'results'
FIGURE_DIR = PROFILE_ROOT / 'figures'
for folder in [WORK_ROOT, PROFILE_ROOT, CACHE_DIR, SPLIT_DIR, CHECKPOINT_DIR, RESULT_DIR, FIGURE_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print('Restored bundle:', RESTORED_BUNDLE)
print('Working directory:', WORK_ROOT)
print('Current run-profile directory:', PROFILE_ROOT)


In [ ]:
# Cell 19 — Save a reproducibility and dataset manifest

accepted_manifest = file_manifest[file_manifest['status'] == 'accepted'].copy()
manifest = {
    'task': 'intra-9' if CFG.protocol == 'intra_subject' else 'cross-9',
    'input_type': 'subject-wise processed PKL features',
    'config': asdict(CFG),
    'dataset_fingerprint': DATASET_FINGERPRINT,
    'accepted_subject_count': NUM_SUBJECTS,
    'subject_file_numbers': subject_file_numbers.tolist(),
    'canonical_subject_shape': [28, 30, 30, 5],
    'flattened_shape': list(data_3d.shape),
    'drop_last_two_channels': DROP_LAST_TWO_CHANNELS,
    'accepted_files': accepted_manifest['path'].tolist(),
    'seed': SEED,
    'torch_version': torch.__version__,
    'device': str(DEVICE),
}
with open(WORK_ROOT / 'run_manifest.json', 'w') as handle:
    json.dump(manifest, handle, indent=2, default=str)
print(json.dumps(manifest, indent=2, default=str)[:3000])


# Cell 20 — Section 3 end — Workspace ready

**Section 3 complete.** Checkpoint folders and automatic restore are ready.


# Cell 21 — Section 4 start — Normalize and prepare two streams

# Section 4 — Paper-style subject normalization and two-stream reconstruction

Expensive arrays are saved separately, so an interrupted Kaggle session can reuse them after restoring the resume bundle.


In [ ]:
# Cell 22 — Normalize each subject and cache the result atomically

NORMALIZED_PATH = CACHE_DIR / f'subject_zscore_{DATASET_FINGERPRINT[:12]}.npy'

if NORMALIZED_PATH.exists():
    normalized_3d = np.load(NORMALIZED_PATH, mmap_mode='r')
    print('Loaded cached normalized data:', NORMALIZED_PATH)
else:
    normalized_array = np.empty_like(data_3d, dtype=np.float32)
    for subject in tqdm(range(NUM_SUBJECTS), desc='Subject z-score'):
        values = data_3d[subject].astype(np.float64, copy=False)
        mean = values.mean(axis=0, keepdims=True)
        std = values.std(axis=0, keepdims=True)
        std[std < 1e-8] = 1.0
        normalized_array[subject] = ((values - mean) / std).astype(np.float32)
    atomic_save_npy(NORMALIZED_PATH, normalized_array)
    del normalized_array
    gc.collect()
    normalized_3d = np.load(NORMALIZED_PATH, mmap_mode='r')
    print('Saved normalized cache:', NORMALIZED_PATH)

assert normalized_3d.shape == EXPECTED_SHAPE
print('Normalized finite:', bool(np.isfinite(normalized_3d).all()))


In [ ]:
# Cell 23 — Build and cache x_F, clip-level x_T, and metadata

CACHE_TAG = DATASET_FINGERPRINT[:12]
X_FREQ_PATH = CACHE_DIR / f'x_frequency_{CACHE_TAG}.npy'
X_TIME_CLIP_PATH = CACHE_DIR / f'x_temporal_by_clip_{CACHE_TAG}.npy'
CLIP_FREQ_PATH = CACHE_DIR / f'clip_frequency_mean_{CACHE_TAG}.npy'
META_PATH = CACHE_DIR / f'sample_metadata_{CACHE_TAG}.npz'

if all(path.exists() for path in [X_FREQ_PATH, X_TIME_CLIP_PATH, CLIP_FREQ_PATH, META_PATH]):
    x_frequency = np.load(X_FREQ_PATH, mmap_mode='r')
    x_temporal_by_clip = np.load(X_TIME_CLIP_PATH, mmap_mode='r')
    clip_frequency_mean = np.load(CLIP_FREQ_PATH, mmap_mode='r')
    meta = np.load(META_PATH)
    sample_labels = meta['labels']
    subject_ids = meta['subjects']
    original_subject_numbers = meta['original_subject_numbers']
    video_ids = meta['videos']
    second_ids = meta['seconds']
    clip_ids = meta['clips']
    print('Loaded cached two-stream tensors and metadata.')
else:
    clips = np.asarray(normalized_3d).reshape(
        NUM_SUBJECTS, NUM_VIDEOS, SECONDS_PER_VIDEO, NUM_NODES, NUM_BANDS
    )
    x_frequency_array = clips.reshape(-1, NUM_NODES, NUM_BANDS).astype(np.float32)
    clip_frequency_array = clips.mean(axis=2).reshape(-1, NUM_NODES, NUM_BANDS).astype(np.float32)
    x_temporal_array = clips.mean(axis=-1).transpose(0, 1, 3, 2).reshape(
        -1, NUM_NODES, SECONDS_PER_VIDEO
    ).astype(np.float32)

    atomic_save_npy(X_FREQ_PATH, x_frequency_array)
    atomic_save_npy(X_TIME_CLIP_PATH, x_temporal_array)
    atomic_save_npy(CLIP_FREQ_PATH, clip_frequency_array)
    np.savez_compressed(
        META_PATH,
        labels=sample_labels,
        subjects=subject_ids,
        original_subject_numbers=original_subject_numbers,
        videos=video_ids,
        seconds=second_ids,
        clips=clip_ids,
    )
    del x_frequency_array, clip_frequency_array, x_temporal_array
    gc.collect()
    x_frequency = np.load(X_FREQ_PATH, mmap_mode='r')
    x_temporal_by_clip = np.load(X_TIME_CLIP_PATH, mmap_mode='r')
    clip_frequency_mean = np.load(CLIP_FREQ_PATH, mmap_mode='r')
    print('Saved two-stream caches.')

print('x_F one-second samples:', x_frequency.shape)
print('x_T unique clips:', x_temporal_by_clip.shape)
print('clip frequency descriptors:', clip_frequency_mean.shape)
print('sample labels:', sample_labels.shape)


In [ ]:
# Cell 24 — Verify processed-stream and sample alignment

last_index = len(sample_labels) - 1
probe = np.unique(np.clip(np.array([0, 29, 30, 839, 840, last_index]), 0, last_index))
alignment = pd.DataFrame({
    'sample': probe,
    'subject_position': subject_ids[probe],
    'subject_file_number': original_subject_numbers[probe],
    'video': video_ids[probe],
    'second': second_ids[probe],
    'clip': clip_ids[probe],
    'label': sample_labels[probe],
    'x_F_shape': [str(x_frequency[index].shape) for index in probe],
    'x_T_shape': [str(x_temporal_by_clip[clip_ids[index]].shape) for index in probe],
})
display(alignment)
assert x_frequency.shape == (NUM_SUBJECTS * 840, 30, 5)
assert x_temporal_by_clip.shape == (NUM_SUBJECTS * 28, 30, 30)
assert clip_ids.max() == x_temporal_by_clip.shape[0] - 1


# Cell 25 — Section 4 end — Streams prepared

**Section 4 complete.** The frequency and temporal streams are cached and aligned.


# Cell 26 — Section 5 start — Heterogeneous adjacency

# Section 5 — Build one graph adjacency per 30-second clip

The official implementation constructs sample-specific adjacency from mutual information. Here, the graph is computed once per clip from its concatenated spatial-spectral and spatial-temporal descriptors, then reused for all 30 seconds of that clip.


In [ ]:
# Cell 27 — Define mutual-information and correlation adjacency functions

def normalized_mutual_information(x: np.ndarray, y: np.ndarray, bins: int = 10) -> float:
    contingency, _, _ = np.histogram2d(x, y, bins=bins)
    if contingency.sum() <= 0:
        return 0.0
    mi = mutual_info_score(None, None, contingency=contingency)
    px = contingency.sum(axis=1)
    py = contingency.sum(axis=0)
    px = px[px > 0] / contingency.sum()
    py = py[py > 0] / contingency.sum()
    hx = -np.sum(px * np.log(px + 1e-12))
    hy = -np.sum(py * np.log(py + 1e-12))
    denominator = math.sqrt(max(hx * hy, 1e-12))
    return float(mi / denominator)

def build_clip_adjacency(features: np.ndarray, method: str, threshold: float, bins: int) -> np.ndarray:
    nodes = features.shape[0]
    adjacency = np.eye(nodes, dtype=np.float32)
    if method == 'correlation':
        matrix = np.corrcoef(features)
        matrix = np.nan_to_num(np.abs(matrix), nan=0.0, posinf=0.0, neginf=0.0)
        matrix[matrix < threshold] = 0.0
        np.fill_diagonal(matrix, 1.0)
        return matrix.astype(np.float32)

    for i in range(nodes):
        for j in range(i + 1, nodes):
            score = normalized_mutual_information(features[i], features[j], bins=bins)
            if score >= threshold:
                adjacency[i, j] = score
                adjacency[j, i] = score
    return adjacency


In [ ]:
# Cell 28 — Build or resume clip adjacency cache

ADJ_PATH = CACHE_DIR / f"clip_adjacency_{DATASET_FINGERPRINT[:12]}_{CFG.adjacency_method}_thr{CFG.adjacency_threshold:.2f}.npy"
ADJ_PROGRESS_PATH = CACHE_DIR / f"clip_adjacency_{DATASET_FINGERPRINT[:12]}_{CFG.adjacency_method}_progress.json"
NUM_CLIPS = NUM_SUBJECTS * NUM_VIDEOS

if ADJ_PATH.exists():
    clip_adjacency = np.load(ADJ_PATH, mmap_mode='r')
    print('Loaded complete adjacency cache:', ADJ_PATH)
else:
    partial_path = CACHE_DIR / f"clip_adjacency_{DATASET_FINGERPRINT[:12]}_{CFG.adjacency_method}_partial.npy"
    if partial_path.exists() and ADJ_PROGRESS_PATH.exists():
        clip_adjacency = np.load(partial_path, mmap_mode='r+')
        with open(ADJ_PROGRESS_PATH) as handle:
            start_clip = int(json.load(handle).get('next_clip', 0))
        print('Resuming adjacency from clip:', start_clip)
    else:
        clip_adjacency = np.lib.format.open_memmap(
            partial_path, mode='w+', dtype=np.float32, shape=(NUM_CLIPS, NUM_NODES, NUM_NODES)
        )
        start_clip = 0

    for clip in tqdm(range(start_clip, NUM_CLIPS), desc='Clip adjacency'):
        combined = np.concatenate(
            [np.asarray(clip_frequency_mean[clip]), np.asarray(x_temporal_by_clip[clip])], axis=-1
        )
        clip_adjacency[clip] = build_clip_adjacency(
            combined,
            method=CFG.adjacency_method,
            threshold=CFG.adjacency_threshold,
            bins=CFG.mi_bins,
        )
        if (clip + 1) % 28 == 0 or clip + 1 == NUM_CLIPS:
            clip_adjacency.flush()
            with open(ADJ_PROGRESS_PATH, 'w') as handle:
                json.dump({'next_clip': clip + 1}, handle)

    del clip_adjacency
    shutil.move(partial_path, ADJ_PATH)
    if ADJ_PROGRESS_PATH.exists():
        ADJ_PROGRESS_PATH.unlink()
    clip_adjacency = np.load(ADJ_PATH, mmap_mode='r')
    print('Saved complete adjacency cache:', ADJ_PATH)

assert clip_adjacency.shape == (NUM_CLIPS, NUM_NODES, NUM_NODES)
density = np.count_nonzero(clip_adjacency) / clip_adjacency.size
print('Adjacency shape:', clip_adjacency.shape)
print('Global non-zero density:', round(float(density), 4))


In [ ]:
# Cell 29 — Visualize one cached adjacency matrix

plt.figure(figsize=(6, 5))
plt.imshow(np.asarray(clip_adjacency[0]), aspect='auto')
plt.title(f'Clip 0 adjacency — {CFG.adjacency_method}')
plt.xlabel('EEG channel')
plt.ylabel('EEG channel')
plt.colorbar()
plt.tight_layout()
plt.show()


# Cell 30 — Section 5 end — Graph cache ready

**Section 5 complete.** Clip-level graph matrices are cached and reusable.


# Cell 31 — Section 6 start — Nested-CV splits

# Section 6 — Build exact outer and inner splits for cross-9

- **Cross-9:** outer and inner folds are subject-disjoint.
- **Intra-9:** folds are assigned by second positions inside every video, ensuring every subject and video contributes to each outer fold without overlapping one-second samples.


In [ ]:
# Cell 32 — Define cross-subject and intra-subject nested-CV splits

def build_cross_subject_splits(seed: int = 42) -> List[Dict]:
    rng = np.random.default_rng(seed)
    shuffled_subjects = rng.permutation(NUM_SUBJECTS)
    if NUM_SUBJECTS == 123:
        test_subject_folds = [shuffled_subjects[index * 12:(index + 1) * 12] for index in range(9)]
        test_subject_folds.append(shuffled_subjects[108:])
    else:
        test_subject_folds = [np.asarray(group, dtype=np.int64) for group in np.array_split(shuffled_subjects, CFG.outer_folds)]

    splits = []
    for outer_fold, test_subjects in enumerate(test_subject_folds):
        train_subjects = np.setdiff1d(shuffled_subjects, test_subjects, assume_unique=True)
        inner_val_groups = np.array_split(train_subjects, CFG.inner_folds)
        inner = []
        for inner_fold, val_subjects in enumerate(inner_val_groups):
            inner_train_subjects = np.setdiff1d(train_subjects, val_subjects, assume_unique=True)
            inner.append({
                'fold': inner_fold,
                'train_idx': np.flatnonzero(np.isin(subject_ids, inner_train_subjects)),
                'val_idx': np.flatnonzero(np.isin(subject_ids, val_subjects)),
                'train_subjects': inner_train_subjects,
                'val_subjects': val_subjects,
            })
        splits.append({
            'outer_fold': outer_fold,
            'train_idx': np.flatnonzero(np.isin(subject_ids, train_subjects)),
            'test_idx': np.flatnonzero(np.isin(subject_ids, test_subjects)),
            'train_subjects': train_subjects,
            'test_subjects': test_subjects,
            'inner': inner,
        })
    return splits

def build_intra_subject_splits() -> List[Dict]:
    # Paper-aligned: the 30 one-second windows of every video and subject
    # are distributed equally across ten folds by second position.
    outer_assignment = second_ids % CFG.outer_folds
    splits = []
    for outer_fold in range(CFG.outer_folds):
        test_mask = outer_assignment == outer_fold
        train_mask = ~test_mask
        remaining_seconds = np.array([
            second for second in range(SECONDS_PER_VIDEO)
            if second % CFG.outer_folds != outer_fold
        ])
        inner_groups = np.array_split(remaining_seconds, CFG.inner_folds)
        inner = []
        for inner_fold, val_seconds in enumerate(inner_groups):
            val_mask = train_mask & np.isin(second_ids, val_seconds)
            inner_train_mask = train_mask & ~val_mask
            inner.append({
                'fold': inner_fold,
                'train_idx': np.flatnonzero(inner_train_mask),
                'val_idx': np.flatnonzero(val_mask),
                'val_seconds': val_seconds,
            })
        splits.append({
            'outer_fold': outer_fold,
            'train_idx': np.flatnonzero(train_mask),
            'test_idx': np.flatnonzero(test_mask),
            'test_seconds': np.flatnonzero(np.arange(SECONDS_PER_VIDEO) % CFG.outer_folds == outer_fold),
            'inner': inner,
        })
    return splits


In [ ]:
# Cell 33 — Load or create a dataset-specific split cache

SPLIT_PATH = SPLIT_DIR / (
    f'{CFG.protocol}_outer{CFG.outer_folds}_inner{CFG.inner_folds}_'
    f'n{NUM_SUBJECTS}_{DATASET_FINGERPRINT[:12]}_seed{CFG.seed}.pkl'
)
if SPLIT_PATH.exists():
    with open(SPLIT_PATH, 'rb') as handle:
        CV_SPLITS = pickle.load(handle)
    print('Loaded cached splits:', SPLIT_PATH)
else:
    CV_SPLITS = (
        build_cross_subject_splits(CFG.seed)
        if CFG.protocol == 'cross_subject'
        else build_intra_subject_splits()
    )
    temporary = SPLIT_PATH.with_suffix('.pkl.tmp')
    with open(temporary, 'wb') as handle:
        pickle.dump(CV_SPLITS, handle, protocol=pickle.HIGHEST_PROTOCOL)
    temporary.replace(SPLIT_PATH)
    print('Saved split cache:', SPLIT_PATH)

assert len(CV_SPLITS) == CFG.outer_folds


In [ ]:
# Cell 34 — Verify split leakage, sizes, and class coverage

split_rows = []
for outer in CV_SPLITS:
    train_idx = outer['train_idx']
    test_idx = outer['test_idx']
    assert np.intersect1d(train_idx, test_idx).size == 0
    assert np.union1d(train_idx, test_idx).size == len(sample_labels)
    if CFG.protocol == 'cross_subject':
        assert np.intersect1d(subject_ids[train_idx], subject_ids[test_idx]).size == 0
    for inner in outer['inner']:
        assert np.intersect1d(inner['train_idx'], inner['val_idx']).size == 0
        assert np.intersect1d(inner['val_idx'], test_idx).size == 0
        assert set(np.unique(sample_labels[inner['val_idx']])) == set(range(NUM_CLASSES))
    split_rows.append({
        'outer_fold': outer['outer_fold'] + 1,
        'train_samples': len(train_idx),
        'test_samples': len(test_idx),
        'train_subjects': len(np.unique(subject_ids[train_idx])),
        'train_subject_file_numbers': len(np.unique(original_subject_numbers[train_idx])),
        'test_subjects': len(np.unique(subject_ids[test_idx])),
        'test_classes': len(np.unique(sample_labels[test_idx])),
    })
split_table = pd.DataFrame(split_rows)
display(split_table)


# Cell 35 — Section 6 end — Splits verified

**Section 6 complete.** All outer and inner folds are cached and leakage checks passed.


# Cell 36 — Section 7 start — Dataset and data loaders

# Section 7 — PyTorch dataset and loaders

The temporal tensor and adjacency are indexed by clip ID, preventing 30-fold duplication in memory.


In [ ]:
# Cell 37 — Define the two-stream dataset

class FACEDHetDataset(Dataset):
    def __init__(self, indices: np.ndarray):
        self.indices = np.asarray(indices, dtype=np.int64)

    def __len__(self) -> int:
        return int(self.indices.size)

    def __getitem__(self, item: int):
        sample_index = int(self.indices[item])
        clip_index = int(clip_ids[sample_index])
        x_f = torch.from_numpy(np.asarray(x_frequency[sample_index], dtype=np.float32).copy())
        x_t = torch.from_numpy(np.asarray(x_temporal_by_clip[clip_index], dtype=np.float32).copy())
        adjacency = torch.from_numpy(np.asarray(clip_adjacency[clip_index], dtype=np.float32).copy())
        label = torch.tensor(int(sample_labels[sample_index]), dtype=torch.long)
        return x_f, x_t, adjacency, label, sample_index

def make_loader(indices: np.ndarray, shuffle: bool, batch_size: Optional[int] = None) -> DataLoader:
    return DataLoader(
        FACEDHetDataset(indices),
        batch_size=batch_size or CFG.batch_size,
        shuffle=shuffle,
        num_workers=CFG.num_workers,
        pin_memory=torch.cuda.is_available(),
        persistent_workers=CFG.num_workers > 0,
        drop_last=False,
    )


In [ ]:
# Cell 38 — Inspect one mini-batch

preview_loader = make_loader(CV_SPLITS[0]['inner'][0]['val_idx'][:1024], shuffle=False, batch_size=16)
batch_x_f, batch_x_t, batch_adj, batch_y, batch_indices = next(iter(preview_loader))
print('x_F batch:', tuple(batch_x_f.shape))
print('x_T batch:', tuple(batch_x_t.shape))
print('adjacency batch:', tuple(batch_adj.shape))
print('labels batch:', tuple(batch_y.shape))
print('sample-index batch:', tuple(batch_indices.shape))


# Cell 39 — Section 7 end — Data loaders ready

**Section 7 complete.**


# Cell 40 — Section 8 start — HetEmotionNet model

# Section 8 — Standalone HetEmotionNet-style architecture

Each stream applies normalized graph propagation followed by a bidirectional GRU. The two stream embeddings are projected, flattened, fused, and classified into nine emotions.


In [ ]:
# Cell 41 — Define GNN4EEG-style graph normalization and graph–BiGRU stream

def normalize_adjacency(adjacency: torch.Tensor) -> torch.Tensor:
    batch, nodes, _ = adjacency.shape
    identity = torch.eye(nodes, device=adjacency.device, dtype=adjacency.dtype).expand(batch, -1, -1)
    off_diagonal = adjacency * (1.0 - identity)
    adjacency_with_loops = off_diagonal + identity
    degree = adjacency_with_loops.sum(dim=-1).clamp_min(1e-6)
    inv_sqrt = degree.pow(-0.5)
    return inv_sqrt.unsqueeze(-1) * adjacency_with_loops * inv_sqrt.unsqueeze(-2)

class GraphBiGRUStream(nn.Module):
    """Standalone equivalent of GNN4EEG STDCN_with_GRU."""

    def __init__(self, num_nodes: int, sequence_length: int):
        super().__init__()
        self.num_nodes = num_nodes
        self.sequence_length = sequence_length
        self.graph_scale = nn.Parameter(torch.tensor(10.0))
        self.gru = nn.GRU(
            input_size=num_nodes,
            hidden_size=num_nodes,
            num_layers=1,
            batch_first=True,
            bidirectional=True,
        )
        self.batch_norm = nn.BatchNorm1d(sequence_length)

    def forward(self, x: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:
        normalized = normalize_adjacency(adjacency)
        graph_features = F.leaky_relu(torch.bmm(normalized, x) * self.graph_scale)
        sequence = graph_features.transpose(1, 2)  # (batch, sequence_length, nodes)
        recurrent, _ = self.gru(sequence)
        recurrent = self.batch_norm(recurrent)
        return recurrent.transpose(1, 2)  # (batch, 2*nodes, sequence_length)


In [ ]:
# Cell 42 — Define the complete nine-class GNN4EEG HetEmotionNet classifier

class HetEmotionNet(nn.Module):
    def __init__(self, hidden_dim: int, dropout: float, num_classes: int = 9):
        super().__init__()
        self.input_dropout = nn.Dropout(dropout)
        self.frequency_stream = GraphBiGRUStream(NUM_NODES, NUM_BANDS)
        self.temporal_stream = GraphBiGRUStream(NUM_NODES, SECONDS_PER_VIDEO)

        projection_dim = SECONDS_PER_VIDEO // 2
        self.frequency_projection = nn.Linear(NUM_BANDS, projection_dim)
        self.temporal_projection = nn.Linear(SECONDS_PER_VIDEO, projection_dim)

        fused_dim = 2 * NUM_NODES * projection_dim * 2
        self.fusion = nn.Linear(fused_dim, hidden_dim)
        self.activation = nn.LeakyReLU()
        self.output = nn.Linear(hidden_dim, num_classes)

    def forward(self, x_f: torch.Tensor, x_t: torch.Tensor, adjacency: torch.Tensor) -> torch.Tensor:
        x_f = self.input_dropout(x_f)
        x_t = self.input_dropout(x_t)
        frequency = self.frequency_stream(x_f, adjacency)
        temporal = self.temporal_stream(x_t, adjacency)
        frequency = self.frequency_projection(frequency).flatten(1)
        temporal = self.temporal_projection(temporal).flatten(1)
        fused = torch.cat([frequency, temporal], dim=1)
        return self.output(self.activation(self.fusion(fused)))


In [ ]:
# Cell 43 — Run a model shape and gradient smoke test

smoke_model = HetEmotionNet(hidden_dim=20, dropout=CFG.dropout, num_classes=NUM_CLASSES).to(DEVICE)
smoke_x_f = batch_x_f[:4].to(DEVICE)
smoke_x_t = batch_x_t[:4].to(DEVICE)
smoke_adj = batch_adj[:4].to(DEVICE)
smoke_y = batch_y[:4].to(DEVICE)
smoke_logits = smoke_model(smoke_x_f, smoke_x_t, smoke_adj)
smoke_loss = F.cross_entropy(smoke_logits, smoke_y)
smoke_loss.backward()
print('Logit shape:', tuple(smoke_logits.shape))
print('Finite loss:', float(smoke_loss.detach().cpu()))
assert smoke_logits.shape == (4, NUM_CLASSES)
del smoke_model, smoke_logits, smoke_loss, smoke_x_f, smoke_x_t, smoke_adj, smoke_y
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()


# Cell 44 — Section 8 end — Model verified

**Section 8 complete.** A forward and backward pass succeeded.


# Cell 45 — Section 9 start — Training and checkpoint utilities

# Section 9 — Training, evaluation, and epoch-level resume

Exact NCV selects hyperparameters and epoch count from the average inner-validation accuracy. No test fold is used during selection.


In [ ]:
# Cell 46 — Define metric and regularization helpers

def regularization_loss(model: nn.Module, l1_weight: float, l2_weight: float) -> torch.Tensor:
    # Mean-normalized penalties keep coefficients comparable across hidden dimensions.
    l1_terms = [parameter.abs().mean() for parameter in model.parameters() if parameter.requires_grad]
    l2_terms = [parameter.pow(2).mean() for parameter in model.parameters() if parameter.requires_grad]
    l1 = torch.stack(l1_terms).mean() if l1_terms else torch.tensor(0.0, device=DEVICE)
    l2 = torch.stack(l2_terms).mean() if l2_terms else torch.tensor(0.0, device=DEVICE)
    return l1_weight * l1 + l2_weight * l2

def calculate_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> Dict[str, float]:
    precision, recall, f1, _ = precision_recall_fscore_support(
        y_true, y_pred, average='macro', zero_division=0
    )
    return {
        'accuracy': float(accuracy_score(y_true, y_pred)),
        'macro_precision': float(precision),
        'macro_recall': float(recall),
        'macro_f1': float(f1),
    }


In [ ]:
# Cell 47 — Define one training epoch and evaluation

def train_one_epoch(model, loader, optimizer, scaler) -> Dict[str, float]:
    model.train()
    running_loss = 0.0
    y_true, y_pred = [], []
    amp_enabled = CFG.mixed_precision and DEVICE.type == 'cuda'

    for x_f, x_t, adjacency, labels, _ in loader:
        x_f = x_f.to(DEVICE, non_blocking=True)
        x_t = x_t.to(DEVICE, non_blocking=True)
        adjacency = adjacency.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        optimizer.zero_grad(set_to_none=True)

        with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
            logits = model(x_f, x_t, adjacency)
            loss = F.cross_entropy(logits, labels)
            loss = loss + regularization_loss(model, CFG.l1_reg, CFG.l2_reg)

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), CFG.gradient_clip)
        scaler.step(optimizer)
        scaler.update()

        running_loss += float(loss.detach().cpu()) * labels.size(0)
        y_true.append(labels.detach().cpu().numpy())
        y_pred.append(logits.argmax(dim=1).detach().cpu().numpy())

    true = np.concatenate(y_true)
    pred = np.concatenate(y_pred)
    metrics = calculate_metrics(true, pred)
    metrics['loss'] = running_loss / len(loader.dataset)
    return metrics

@torch.no_grad()
def evaluate_model(model, loader, return_predictions: bool = False):
    model.eval()
    running_loss = 0.0
    y_true, y_pred, probabilities, sample_indices = [], [], [], []
    amp_enabled = CFG.mixed_precision and DEVICE.type == 'cuda'

    for x_f, x_t, adjacency, labels, indices in loader:
        x_f = x_f.to(DEVICE, non_blocking=True)
        x_t = x_t.to(DEVICE, non_blocking=True)
        adjacency = adjacency.to(DEVICE, non_blocking=True)
        labels = labels.to(DEVICE, non_blocking=True)
        with torch.autocast(device_type=DEVICE.type, enabled=amp_enabled):
            logits = model(x_f, x_t, adjacency)
            loss = F.cross_entropy(logits, labels)

        running_loss += float(loss.detach().cpu()) * labels.size(0)
        probs = torch.softmax(logits, dim=1)
        y_true.append(labels.detach().cpu().numpy())
        y_pred.append(logits.argmax(dim=1).detach().cpu().numpy())
        probabilities.append(probs.detach().cpu().numpy())
        sample_indices.append(indices.numpy())

    true = np.concatenate(y_true)
    pred = np.concatenate(y_pred)
    metrics = calculate_metrics(true, pred)
    metrics['loss'] = running_loss / len(loader.dataset)
    if return_predictions:
        return metrics, true, pred, np.concatenate(probabilities), np.concatenate(sample_indices)
    return metrics


In [ ]:
# Cell 48 — Define checkpoint save and resume helpers

def atomic_torch_save(payload: Dict, path: Path) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + '.tmp')
    torch.save(payload, temporary)
    temporary.replace(path)

def safe_torch_load(path: Path, map_location):
    try:
        return torch.load(path, map_location=map_location, weights_only=False)
    except TypeError:
        return torch.load(path, map_location=map_location)

def fit_with_resume(
    run_dir: Path,
    train_indices: np.ndarray,
    val_indices: Optional[np.ndarray],
    hidden_dim: int,
    learning_rate: float,
    epochs: int,
) -> Tuple[pd.DataFrame, Path]:
    run_dir.mkdir(parents=True, exist_ok=True)
    last_path = run_dir / 'last.pt'
    history_path = run_dir / 'history.csv'
    complete_path = run_dir / 'complete.json'

    model = HetEmotionNet(hidden_dim, CFG.dropout, CFG.num_classes).to(DEVICE)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    scaler = torch.amp.GradScaler('cuda', enabled=CFG.mixed_precision and DEVICE.type == 'cuda')
    start_epoch = 1
    history = []

    if history_path.exists():
        history = pd.read_csv(history_path).to_dict('records')
    if last_path.exists():
        checkpoint = safe_torch_load(last_path, DEVICE)
        model.load_state_dict(checkpoint['model_state'])
        optimizer.load_state_dict(checkpoint['optimizer_state'])
        if checkpoint.get('scaler_state'):
            scaler.load_state_dict(checkpoint['scaler_state'])
        start_epoch = int(checkpoint['epoch']) + 1
        print(f'Resuming {run_dir.name} from epoch {start_epoch}.')

    if complete_path.exists() and start_epoch > epochs:
        return pd.DataFrame(history), last_path

    train_loader = make_loader(train_indices, shuffle=True)
    val_loader = make_loader(val_indices, shuffle=False) if val_indices is not None else None

    for epoch in range(start_epoch, epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, scaler)
        val_metrics = evaluate_model(model, val_loader) if val_loader is not None else {}
        row = {'epoch': epoch, **{f'train_{k}': v for k, v in train_metrics.items()}}
        row.update({f'val_{k}': v for k, v in val_metrics.items()})
        history = [item for item in history if int(item['epoch']) != epoch] + [row]
        history_df = pd.DataFrame(history).sort_values('epoch')
        history_df.to_csv(history_path, index=False)

        atomic_torch_save({
            'epoch': epoch,
            'model_state': model.state_dict(),
            'optimizer_state': optimizer.state_dict(),
            'scaler_state': scaler.state_dict(),
            'hidden_dim': hidden_dim,
            'learning_rate': learning_rate,
            'config': asdict(CFG),
        }, last_path)

        message = f"Epoch {epoch:03d}/{epochs} | train acc {train_metrics['accuracy']:.4f}"
        if val_loader is not None:
            message += f" | val acc {val_metrics['accuracy']:.4f} | val F1 {val_metrics['macro_f1']:.4f}"
        print(message)

    with open(complete_path, 'w') as handle:
        json.dump({'epochs': epochs, 'completed': True}, handle, indent=2)
    return pd.DataFrame(history).sort_values('epoch'), last_path


# Cell 49 — Section 9 end — Training utilities ready

**Section 9 complete.**


# Cell 50 — Section 10 start — Nested cross-validation runner

# Section 10 — Resumable NCV orchestration

For each outer fold:

1. train every `(hidden_dim, learning_rate)` pair on each of three inner folds;
2. average inner-validation accuracy at every epoch;
3. select the hyperparameters and epoch with highest mean inner accuracy;
4. initialize a fresh model on the complete outer-training set;
5. train for the selected epoch count;
6. evaluate once on the untouched outer test set.


In [ ]:
# Cell 51 — Define inner-grid selection

def select_hyperparameters_for_outer_fold(outer: Dict) -> Dict:
    outer_number = outer['outer_fold'] + 1
    selection_path = RESULT_DIR / f'outer_{outer_number:02d}_selection.json'
    if selection_path.exists():
        with open(selection_path) as handle:
            selected = json.load(handle)
        print('Loaded completed selection:', selection_path)
        return selected

    candidate_rows = []
    for hidden_dim, learning_rate in product(CFG.effective_hidden_grid, CFG.effective_lr_grid):
        inner_histories = []
        for inner in outer['inner']:
            run_dir = (
                CHECKPOINT_DIR / f'outer_{outer_number:02d}' /
                f'h{hidden_dim}_lr{learning_rate:g}' / f"inner_{inner['fold'] + 1:02d}"
            )
            history, _ = fit_with_resume(
                run_dir=run_dir,
                train_indices=inner['train_idx'],
                val_indices=inner['val_idx'],
                hidden_dim=hidden_dim,
                learning_rate=learning_rate,
                epochs=CFG.effective_max_epochs,
            )
            inner_histories.append(history[['epoch', 'val_accuracy', 'val_macro_f1']].copy())

        merged = inner_histories[0].rename(columns={
            'val_accuracy': 'accuracy_0', 'val_macro_f1': 'f1_0'
        })
        for index, history in enumerate(inner_histories[1:], start=1):
            renamed = history.rename(columns={
                'val_accuracy': f'accuracy_{index}', 'val_macro_f1': f'f1_{index}'
            })
            merged = merged.merge(renamed, on='epoch', how='inner')

        accuracy_columns = [column for column in merged if column.startswith('accuracy_')]
        f1_columns = [column for column in merged if column.startswith('f1_')]
        merged['mean_val_accuracy'] = merged[accuracy_columns].mean(axis=1)
        merged['mean_val_macro_f1'] = merged[f1_columns].mean(axis=1)
        best_row = merged.sort_values(
            ['mean_val_accuracy', 'mean_val_macro_f1', 'epoch'], ascending=[False, False, True]
        ).iloc[0]
        candidate_rows.append({
            'hidden_dim': hidden_dim,
            'learning_rate': learning_rate,
            'selected_epoch': int(best_row['epoch']),
            'mean_val_accuracy': float(best_row['mean_val_accuracy']),
            'mean_val_macro_f1': float(best_row['mean_val_macro_f1']),
        })
        merged.to_csv(
            RESULT_DIR / f'outer_{outer_number:02d}_h{hidden_dim}_lr{learning_rate:g}_inner_curve.csv',
            index=False,
        )

    candidates = pd.DataFrame(candidate_rows).sort_values(
        ['mean_val_accuracy', 'mean_val_macro_f1'], ascending=False
    )
    candidates.to_csv(RESULT_DIR / f'outer_{outer_number:02d}_candidate_summary.csv', index=False)
    selected = candidates.iloc[0].to_dict()
    selected['hidden_dim'] = int(selected['hidden_dim'])
    selected['selected_epoch'] = int(selected['selected_epoch'])
    with open(selection_path, 'w') as handle:
        json.dump(selected, handle, indent=2)
    display(candidates)
    return selected


In [ ]:
# Cell 52 — Define final outer-fold training and testing

def run_final_outer_fold(outer: Dict, selected: Dict) -> Dict:
    outer_number = outer['outer_fold'] + 1
    metrics_path = RESULT_DIR / f'outer_{outer_number:02d}_test_metrics.json'
    prediction_path = RESULT_DIR / f'outer_{outer_number:02d}_predictions.csv'
    final_dir = CHECKPOINT_DIR / f'outer_{outer_number:02d}' / 'final_model'

    if metrics_path.exists() and prediction_path.exists():
        with open(metrics_path) as handle:
            print('Loaded completed outer result:', metrics_path)
            return json.load(handle)

    history, checkpoint_path = fit_with_resume(
        run_dir=final_dir,
        train_indices=outer['train_idx'],
        val_indices=None,
        hidden_dim=int(selected['hidden_dim']),
        learning_rate=float(selected['learning_rate']),
        epochs=int(selected['selected_epoch']),
    )

    model = HetEmotionNet(int(selected['hidden_dim']), CFG.dropout, CFG.num_classes).to(DEVICE)
    checkpoint = safe_torch_load(checkpoint_path, DEVICE)
    model.load_state_dict(checkpoint['model_state'])
    test_loader = make_loader(outer['test_idx'], shuffle=False)
    metrics, true, pred, probabilities, indices = evaluate_model(model, test_loader, return_predictions=True)

    metrics.update({
        'outer_fold': outer_number,
        'hidden_dim': int(selected['hidden_dim']),
        'learning_rate': float(selected['learning_rate']),
        'selected_epoch': int(selected['selected_epoch']),
        'test_samples': int(len(indices)),
    })
    with open(metrics_path, 'w') as handle:
        json.dump(metrics, handle, indent=2)

    prediction_frame = pd.DataFrame({
        'sample_index': indices,
        'subject_position': subject_ids[indices],
        'subject_file_number': original_subject_numbers[indices],
        'video': video_ids[indices],
        'second': second_ids[indices],
        'clip_id': clip_ids[indices],
        'true_label': true,
        'predicted_label': pred,
        'true_emotion': [EMOTION_NAMES[i] for i in true],
        'predicted_emotion': [EMOTION_NAMES[i] for i in pred],
    })
    for class_id, emotion in enumerate(EMOTION_NAMES):
        prediction_frame[f'prob_{class_id}_{emotion.lower()}'] = probabilities[:, class_id]
    prediction_frame.to_csv(prediction_path, index=False)
    return metrics


In [ ]:
# Cell 53 — Run all scheduled outer folds with automatic skipping/resume

fold_metrics = []
for fold_index in CFG.effective_folds:
    outer = CV_SPLITS[int(fold_index)]
    print('\n' + '=' * 90)
    print(f"OUTER FOLD {int(fold_index) + 1}/{CFG.outer_folds}")
    print('=' * 90)
    selected = select_hyperparameters_for_outer_fold(outer)
    print('Selected configuration:', selected)
    metrics = run_final_outer_fold(outer, selected)
    fold_metrics.append(metrics)
    display(pd.DataFrame([metrics]))
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

fold_metrics_df = pd.DataFrame(fold_metrics).sort_values('outer_fold')
fold_metrics_df.to_csv(RESULT_DIR / 'fold_metrics_current_session.csv', index=False)
display(fold_metrics_df)


# Cell 54 — Section 10 end — Scheduled folds complete

**Section 10 complete.** Completed folds are saved and will be skipped during the next resumed session.


# Cell 55 — Section 11 start — Results, inference, and export

# Section 11 — Aggregate completed results and create the next resume bundle

This section works even when only part of the 10-fold experiment is complete.


In [ ]:
# Cell 56 — Collect every completed outer-fold result

completed_metrics = []
for metrics_file in sorted(RESULT_DIR.glob('outer_*_test_metrics.json')):
    with open(metrics_file) as handle:
        completed_metrics.append(json.load(handle))

if not completed_metrics:
    raise RuntimeError('No completed outer-fold metrics were found.')

completed_df = pd.DataFrame(completed_metrics).sort_values('outer_fold')
completed_df.to_csv(RESULT_DIR / 'all_completed_fold_metrics.csv', index=False)
display(completed_df)

metric_columns = ['accuracy', 'macro_precision', 'macro_recall', 'macro_f1']
summary_rows = []
for metric in metric_columns:
    summary_rows.append({
        'metric': metric,
        'mean': completed_df[metric].mean(),
        'std': completed_df[metric].std(ddof=1) if len(completed_df) > 1 else np.nan,
        'minimum': completed_df[metric].min(),
        'maximum': completed_df[metric].max(),
        'completed_folds': len(completed_df),
    })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(RESULT_DIR / 'aggregate_metrics.csv', index=False)
display(summary_df)


In [ ]:
# Cell 57 — Combine predictions and plot the confusion matrix

prediction_files = sorted(RESULT_DIR.glob('outer_*_predictions.csv'))
all_predictions = pd.concat([pd.read_csv(path) for path in prediction_files], ignore_index=True)
all_predictions.to_csv(RESULT_DIR / 'all_completed_predictions.csv', index=False)

matrix = confusion_matrix(
    all_predictions['true_label'], all_predictions['predicted_label'], labels=np.arange(NUM_CLASSES)
)
normalized_matrix = matrix / np.maximum(matrix.sum(axis=1, keepdims=True), 1)

plt.figure(figsize=(10, 8))
plt.imshow(normalized_matrix, vmin=0, vmax=1, aspect='auto')
plt.title(f"HetEmotionNet {'intra-9' if CFG.protocol == 'intra_subject' else 'cross-9'} — normalized confusion matrix")
plt.xticks(np.arange(NUM_CLASSES), EMOTION_NAMES, rotation=45, ha='right')
plt.yticks(np.arange(NUM_CLASSES), EMOTION_NAMES)
plt.xlabel('Predicted class')
plt.ylabel('True class')
plt.colorbar(label='Row-normalized proportion')
plt.tight_layout()
confusion_path = FIGURE_DIR / 'normalized_confusion_matrix.png'
plt.savefig(confusion_path, dpi=200, bbox_inches='tight')
plt.show()

print(classification_report(
    all_predictions['true_label'],
    all_predictions['predicted_label'],
    labels=np.arange(NUM_CLASSES),
    target_names=EMOTION_NAMES,
    zero_division=0,
))


In [ ]:
# Cell 58 — Plot completed fold-wise performance

plt.figure(figsize=(9, 5))
plt.plot(completed_df['outer_fold'], completed_df['accuracy'], marker='o', label='Accuracy')
plt.plot(completed_df['outer_fold'], completed_df['macro_f1'], marker='o', label='Macro F1')
plt.xlabel('Outer fold')
plt.ylabel('Score')
plt.ylim(0, 1)
plt.title('Completed outer-fold performance')
plt.grid(alpha=0.3)
plt.legend()
plt.tight_layout()
performance_path = FIGURE_DIR / 'fold_performance.png'
plt.savefig(performance_path, dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Cell 59 — Run inference on one saved test sample

best_row = completed_df.sort_values(['accuracy', 'macro_f1'], ascending=False).iloc[0]
best_fold = int(best_row['outer_fold'])
best_metrics_path = RESULT_DIR / f'outer_{best_fold:02d}_test_metrics.json'
with open(best_metrics_path) as handle:
    best_metrics = json.load(handle)

model_path = CHECKPOINT_DIR / f'outer_{best_fold:02d}' / 'final_model' / 'last.pt'
best_model = HetEmotionNet(int(best_metrics['hidden_dim']), CFG.dropout, CFG.num_classes).to(DEVICE)
state = safe_torch_load(model_path, DEVICE)
best_model.load_state_dict(state['model_state'])
best_model.eval()

sample_row = all_predictions[all_predictions['sample_index'].notna()].iloc[0]
sample_index = int(sample_row['sample_index'])
clip_index = int(clip_ids[sample_index])
with torch.no_grad():
    logits = best_model(
        torch.from_numpy(np.asarray(x_frequency[sample_index]).copy()).unsqueeze(0).to(DEVICE),
        torch.from_numpy(np.asarray(x_temporal_by_clip[clip_index]).copy()).unsqueeze(0).to(DEVICE),
        torch.from_numpy(np.asarray(clip_adjacency[clip_index]).copy()).unsqueeze(0).to(DEVICE),
    )
    probability = torch.softmax(logits, dim=1).cpu().numpy()[0]
prediction = int(probability.argmax())
print('Sample index:', sample_index)
print('True emotion:', EMOTION_NAMES[int(sample_labels[sample_index])])
print('Predicted emotion:', EMOTION_NAMES[prediction])
display(pd.DataFrame({'emotion': EMOTION_NAMES, 'probability': probability}).sort_values('probability', ascending=False))


In [ ]:
# Cell 60 — Create a complete Kaggle resume bundle

resume_name = 'hetemotionnet_processed_FACED_cross_9_full_resumable_resume_bundle.zip'
resume_path = (Path('/kaggle/working') / resume_name) if Path('/kaggle/working').exists() else (WORK_ROOT.parent / resume_name)

if resume_path.exists():
    resume_path.unlink()
with zipfile.ZipFile(resume_path, 'w', compression=zipfile.ZIP_DEFLATED, allowZip64=True) as archive:
    for file_path in WORK_ROOT.rglob('*'):
        if file_path.is_file():
            archive.write(file_path, arcname=str(file_path.relative_to(WORK_ROOT.parent)))

print('Resume bundle created:', resume_path)
print('Bundle size (GB):', round(resume_path.stat().st_size / 1024**3, 4))
print('Download this ZIP and attach it as a private Kaggle dataset in the next session.')


# Cell 61 — Notebook complete

The `cross-9` processed-PKL HetEmotionNet workflow is complete. Cached preprocessing, graph progress, splits, epoch checkpoints, completed folds, metrics, predictions, plots, and the resume bundle are stored under the task workspace.
